In [1]:
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment
import random

c:\Users\jiova\anaconda3\envs\speech_recognition\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [3]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

In [2]:
audio_file = "./sample_audio.wav"
audio, sr = librosa.load(audio_file, sr=None)

c:\Users\jiova\anaconda3\envs\speech_recognition\lib\site-packages\librosa\core\intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [ ]:
def predict_text():
    waveform, sr = torchaudio.load(audio_file)
    waveform = waveform.mean(dim=0, keepdim=True)

    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(waveform)

    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission:torch.Tensor)->str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return ''.join([self.labels[i] for i in indices])

    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript


In [5]:
def export_func(audio):
    while True:
        file_name = input(" file name >> ")
        if file_name.endswith(".wav"):
            break
        else:
            print("must be a .wav file")

    sf.write(file_name, audio, sr)
    print("saved")

In [6]:
def pitch_shift():
    while True:
        pitch_rate = float(input("pitch rate:"))
        if pitch_rate > -5 and pitch_rate <5:
            break
        else:
            print("pitch msut be between -5 and 5")

    res = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_rate)
    export_func(res)

def white_noise(audio, value):
    noise = np.random.normal(0, audio.std(), audio.size)
    return audio + noise * value



def add_white_noise():
    while True:
        value = float(input("noise value:"))
        if value > 0.1 and value < 0.5:
            break
        else:
            print("invalid noise must be 0.1<x<0.5")

    result = white_noise(audio, value)
    export_func(result)

In [7]:
def export_pydub(audio):
    while True:
        file_name = input(" file name >> ")
        if file_name.endswith(".wav"):
            break
        else:
            print("must be a .wav file")

    audio.export(file_name, format="wav")
    print("exported")

In [8]:
def repeat_clip(clip):
    while True:
        value = int(input("noise value:"))
        if value > 0:
            break
        else:
            print("> 0")
    result = clip * value
    export_pydub(result)

def fade_in(clip):
    while True:
        value = int(input("noise value:"))
        if value > 0:
            break
        else:
            print("> 0")

    result = clip.fade_in(value)
    export_pydub(result)

In [9]:
def menu1():
    clip = AudioSegment.from_wav(audio_file)
    while(True):
        print()
        print("1. Pitch shift")
        print("2. Add white noise")
        print("3. Repeat clip")
        print("4. Insert fade in")
        print("5. Back to main menu")
        choice = input("Choose: ")

        if choice == "1":
            pitch_shift()
        elif choice == "2":
            add_white_noise()
        elif choice == "3":
            repeat_clip(clip)
        elif choice == "4":
            fade_in(clip)
        elif choice == "5":
            break
        else:
            print("Invalid choice.")

In [10]:
def main_menu():
    while(True):
        print()
        print("1. Audio Augmentation")
        print("2. Predict Text")
        print("3. Exit")
        choice = (input("Choose: "))

        if choice =="1":
            menu1()
        elif choice =="2":
            transcript=predict_text()
            print(transcript)
        elif choice =="3":
            break
        else:
            print("invalid input")

    print("byee")

In [11]:
main_menu()


1. Audio Augmentation
2. Predict Text
3. Exit
invalid input

1. Audio Augmentation
2. Predict Text
3. Exit

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
invalid noise must be 0.1<x<0.5
invalid noise must be 0.1<x<0.5
must be a .wav file
saved

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
Invalid choice.

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
must be a .wav file
must be a .wav file
exported

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
Invalid choice.

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
must be a .wav file
must be a .wav file
exported

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
Invalid choice.

1. Pitch shift
2. Add white noise
3. Repeat clip
4. Insert fade in
5. Back to main menu
must be a .wav file
exported

1. 